In [ ]:
import io
import json
from pathlib import Path

import httpx
from PIL import Image

BASE_URL = "http://localhost:8003"

RESUME_BODY = {
    "rules": [
        {
            "id": "rule_4",
            "condition": "Резюме должно быть заполнено на русском языке, допускаются иностранные термины для описания профессиональных навыков и других особых случаев, данные на английском допустимы в пунктах Контактная информация, Близкие родственники, Образование, Навыки работы с компьютером, Опыт работы"
        },
        {
            "id": "rule_7",
            "condition": "Информация не должна содержать слова и выражения, не соответствующие нормам современного русского языка, допустимы стилистические неточности, но главное, чтобы не было матов и токсичности"
        }
    ],
    "resume": {
        "fullname": "Шилоносов Владимир Андреевич",
        "fullnameChange": "До 2019 года — Иванов Владимир Андреевич, смена фамилии при вступлении в брак",
        "citizenship": "Российская Федерация",
        "passportOrEquivalent": "45 01 №123456, выдан УФМС по г. Москве 12.05.2016",
        "snils": "123-456 00 11",
        "birthdate": "1998-07-22",
        "placeOfBirth": "г. Санкт-Петербург",
        "registrationAddress": "г. Санкт-Петербург, ул. Невский пр., д. 12, кв. 34",
        "actualResidenceAddress": "г. Санкт-Петербург, ул. Ленина, д. 5",
        "contactInformation": "+7 (921) 123-45-67, vladimir@mail.ru",
        "closeRelatives": [
            {
                "relationship": "Отец",
                "fullname": "Шилоносов Андрей Петрович",
                "birthdate": "1968-05-22",
                "job": "АО «Газпром», инженер",
                "address": "г. Санкт-Петербург, ул. Ленина, д. 15, кв. 8"
            }
        ],
        "education": {
            "higherEducation": [
                {
                    "dateOfAdmission": "2016-09-01",
                    "dateOfGraduation": "2020-06-30",
                    "institutionName": "Санкт-Петербургский государственный университет",
                    "specialty": "01.03.02 Прикладная математика и информатика",
                    "level": "Бакалавриат",
                    "formOfEducation": "Очная",
                    "year": 4,
                    "haveDiploma": True,
                    "educationFilename": "__FILENAME__"
                }
            ],
            "additionalEducation": [
                {
                    "dateOfAdmission": "2016-09-01",
                    "dateOfGraduation": "2020-06-30",
                    "institutionName": "Санкт-Петербургский государственный университет",
                    "educationalProgram": "Data Science и машинное обучение",
                    "programType": "Повышение квалификации",
                    "hoursИumber": 144
                }
            ],
            "postgraduate": [
                {
                    "dateOfAdmission": "2016-09-01",
                    "dateOfGraduation": "2020-06-30",
                    "institutionName": "Санкт-Петербургский государственный университет",
                    "specialty": "Информационные технологии",
                    "degree": "Кандидат технических наук",
                    "scienceBranch": "Технические науки"
                }
            ]
        },
        "languges": [{"name": "Английский", "level": "SpeakFluently"}],
        "softwareSkills": [{"type": "Текстовые редакторы", "nameOfProduct": "Microsoft Word", "level": "Fluent"}],
        "publications": ["Применение ML в госуправлении"],
        "awards": ["Премия губернатора за успехи в учебе"],
        "militaryLiable": True,
        "militaryСategory": "Годен к военной службе",
        "professionalInterests": "Анализ данных, финансы, образование",
        "additionalInfo": "Ответственный, коммуникабельный, интересуюсь IT и саморазвитием",
        "motivation": "Хочу внести вклад в развитие Санкт-Петербурга и реализовать свой потенциал",
        "source": "Информация в вузе (центр карьеры, ярмарка вакансий)"
    }
}

DIPLOMAS_DIR = Path("Примеры дипломов и справок")
NOT_OK_DIR = DIPLOMAS_DIR / "Не ок"
OK_DIR = DIPLOMAS_DIR / "Ок"

print("Не ок:", len(list(NOT_OK_DIR.iterdir())))
print("Ок:", len(list(OK_DIR.iterdir())))

Не ок: 21
Ок: 17


In [2]:
def image_to_pdf_bytes(image_path: Path) -> bytes:
    """Конвертирует JPG/PNG в PDF в памяти."""
    img = Image.open(image_path).convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="PDF")
    return buf.getvalue()


def run_test(image_path: Path) -> dict:
    """Загружает файл и запускает модерацию. Возвращает результат."""
    pdf_bytes = image_to_pdf_bytes(image_path)
    pdf_filename = image_path.stem + ".pdf"

    # 1. Upload
    with httpx.Client(base_url=BASE_URL, timeout=120) as client:
        upload_resp = client.post(
            "/moderator/reserve/upload-education-file",
            files={"file": (pdf_filename, pdf_bytes, "application/pdf")},
        )
        upload_resp.raise_for_status()
        saved_filename = upload_resp.json()["educationFilename"]

        # 2. Selection
        body = json.loads(json.dumps(RESUME_BODY))
        body["resume"]["education"]["higherEducation"][0]["educationFilename"] = saved_filename

        sel_resp = client.post("/moderator/reserve/selection", json=body)
        sel_resp.raise_for_status()
        return sel_resp.json()


def print_result(image_path: Path, result: dict, expected_ok: bool) -> None:
    edu = result.get("educationInfo", [{}])
    resolution = edu[0].get("resolution", {}) if edu else {}
    valid = resolution.get("valid")
    reason = resolution.get("noValidReason", "—")
    overall = result.get("result", {}).get("overallSuccess")
    match = "✓" if (valid == expected_ok) else "✗ НЕОЖИДАННО"
    print(f"  {match} {image_path.name}")
    print(f"     valid={valid}  noValidReason={reason}  overallSuccess={overall}")

In [3]:
print("=== НЕ ОК ===")
for img_path in sorted(NOT_OK_DIR.iterdir()):
    try:
        result = run_test(img_path)
        print_result(img_path, result, expected_ok=False)
    except Exception as e:
        print(f"  ERROR {img_path.name}: {e}")

=== НЕ ОК ===
  ERROR Диплом ДА, направление НЕТ.jpg: [Errno 61] Connection refused
  ERROR Диплом ДА, направление НЕТ_1.jpg: [Errno 61] Connection refused
  ERROR Диплом ДА, направление НЕТ_2.jpg: [Errno 61] Connection refused
  ERROR Диплом ДА, направление НЕТ_6.jpg: [Errno 61] Connection refused
  ERROR Диплом ДА, направление НЕТ_7.jpg: [Errno 61] Connection refused
  ERROR Еще одна переподготовка.jpg: [Errno 61] Connection refused
  ERROR И еще переподготовка_1.jpg: [Errno 61] Connection refused
  ERROR И еще переподготовка_2.jpg: [Errno 61] Connection refused
  ERROR И еще переподготовка_3.jpg: [Errno 61] Connection refused
  ERROR НЕ справка и не диплом.jpg: [Errno 61] Connection refused
  ERROR Не вышка. Это диплом об ССО_1.jpg: [Errno 61] Connection refused
  ERROR Не полноценный диплом (проект)_1.jpg: [Errno 61] Connection refused
  ERROR Не полноценный диплом (проект)_2.jpg: [Errno 61] Connection refused
  ERROR Не справка и не диплом (2).jpg: [Errno 61] Connection refused
  

In [ ]:
print("=== ОК ===")
for img_path in sorted(OK_DIR.iterdir()):
    try:
        result = run_test(img_path)
        print_result(img_path, result, expected_ok=True)
    except Exception as e:
        print(f"  ERROR {img_path.name}: {e}")